# 02 — Préparation des données
**Projet IA-DS-2026-01 — Détection de fraude bancaire**

Ce notebook réalise :
1. Le nettoyage (suppression des doublons)
2. Le feature engineering temporel (`TransactionHour`, `TransactionDay`)
3. Le **découpage temporel strict** 70/15/15 (⚠️ jamais aléatoire)
4. La standardisation (StandardScaler **fit sur train uniquement**)
5. La sauvegarde des splits dans `data/processed/`

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src import config, data_loader, features, preprocessing


## 1. Chargement et nettoyage

In [2]:
df = data_loader.load_raw_data()
print(f"Avant nettoyage : {df.shape}")

df = preprocessing.clean_data(df)
print(f"Après nettoyage : {df.shape}")


2026-10-08 21:44:51,628 [INFO] Chargement du dataset local : C:\Users\HP\Desktop\projet-IA-DS-2026-01-Moussaoui-Lina\data\raw\creditcard.csv
2026-10-08 21:44:58,235 [INFO] Dataset chargé : 284807 lignes, 31 colonnes
2026-10-08 21:44:58,311 [INFO] Aucune valeur manquante détectée.


Avant nettoyage : (284807, 31)


2026-10-08 21:45:01,927 [INFO] Doublons supprimés : 1081 (sur 284807 lignes initiales)


Après nettoyage : (283726, 31)


## 2. Découpage temporel strict (70/15/15)

⚠️ **INTERDIT** : `train_test_split` aléatoire sur ces données.
On trie par `Time` croissant puis on découpe successivement en train / val / test,
pour que le modèle ne soit jamais entraîné sur des transactions postérieures à
celles utilisées pour l'évaluer — condition nécessaire à une estimation réaliste
de la performance en production.

In [3]:
train_df, val_df, test_df = data_loader.temporal_split(df, ratios=config.SPLIT_RATIOS)

print(f"Train : {len(train_df)} lignes | Time in [{train_df[config.TIME_COL].min():.0f}, {train_df[config.TIME_COL].max():.0f}]")
print(f"Val   : {len(val_df)} lignes | Time in [{val_df[config.TIME_COL].min():.0f}, {val_df[config.TIME_COL].max():.0f}]")
print(f"Test  : {len(test_df)} lignes | Time in [{test_df[config.TIME_COL].min():.0f}, {test_df[config.TIME_COL].max():.0f}]")

print("\nTaux de fraude par split :")
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"  {name}: {100 * d[config.TARGET_COL].mean():.4f}%")


2026-10-08 21:45:02,316 [INFO] Split temporel -> train: 198608 (70.0%) | val: 42558 (15.0%) | test: 42560 (15.0%)
2026-10-08 21:45:02,321 [INFO] Taux de fraude -> train: 0.1843% | val: 0.1292% | test: 0.1222%


Train : 198608 lignes | Time in [0, 132906]
Val   : 42558 lignes | Time in [132906, 151320]
Test  : 42560 lignes | Time in [151320, 172792]

Taux de fraude par split :
  train: 0.1843%
  val: 0.1292%
  test: 0.1222%


**Remarque méthodologique** : le taux de fraude peut légèrement varier d'un split
à l'autre puisque le découpage est temporel et non stratifié — c'est attendu et
acceptable (voire réaliste), contrairement à un split aléatoire qui l'aurait
artificiellement égalisé.

## 3. Feature engineering temporel

In [4]:
train_df = features.create_time_features(train_df)
val_df = features.create_time_features(val_df)
test_df = features.create_time_features(test_df)

train_df[["Time", "TransactionHour", "TransactionDay"]].head()


,Time,TransactionHour,TransactionDay
0,0.0,0.0,0.0
1,0.0,0.0,0.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,2.0,0.0,0.0


## 4. Standardisation (fit sur train uniquement)

⚠️ Le `StandardScaler` est ajusté **exclusivement** sur `X_train`, puis appliqué
tel quel à `X_val`/`X_test`, afin d'éviter toute fuite d'information (leakage)
des statistiques de validation/test vers l'entraînement.

In [5]:
X_train, y_train = features.select_features(train_df)
X_val, y_val = features.select_features(val_df)
X_test, y_test = features.select_features(test_df)

X_train_scaled, X_val_scaled, X_test_scaled, scaler = preprocessing.scale_features(
    X_train, X_val, X_test
)

X_train_scaled.head()


2026-10-08 21:45:06,444 [INFO] Standardisation appliquée sur les colonnes : ['Time', 'Amount', 'TransactionHour', 'TransactionDay'] (fit sur train uniquement)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V22,V23,V24,V25,V26,V27,V28,Amount,TransactionHour,TransactionDay
0,-2.041989,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,0.239843,-2.06273,-0.613975
1,-2.041989,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,-0.349866,-2.06273,-0.613975
2,-2.041960,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,1.159105,-2.06273,-0.613975
3,-2.041960,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,0.135010,-2.06273,-0.613975
4,-2.041931,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,-0.079755,-2.06273,-0.613975


## 5. Sauvegarde des splits prétraités

On sauvegarde les DataFrames **avant** standardisation (au format parquet) : la
standardisation dépendant d'un scaler ajusté sur le train, on préfère la
recalculer explicitement au moment de la modélisation (notebook 03) plutôt que
de figer des valeurs déjà transformées — cela garde le pipeline traçable et
évite de dupliquer un scaler non versionné.

In [6]:
data_loader.save_processed(train_df, config.TRAIN_PATH)
data_loader.save_processed(val_df, config.VAL_PATH)
data_loader.save_processed(test_df, config.TEST_PATH)

print("✅ Splits sauvegardés dans data/processed/")


2026-10-08 21:45:11,445 [INFO] Données sauvegardées : C:\Users\HP\Desktop\projet-IA-DS-2026-01-Moussaoui-Lina\data\processed\train.parquet (198608 lignes)
2026-10-08 21:45:12,593 [INFO] Données sauvegardées : C:\Users\HP\Desktop\projet-IA-DS-2026-01-Moussaoui-Lina\data\processed\val.parquet (42558 lignes)
2026-10-08 21:45:13,726 [INFO] Données sauvegardées : C:\Users\HP\Desktop\projet-IA-DS-2026-01-Moussaoui-Lina\data\processed\test.parquet (42560 lignes)


✅ Splits sauvegardés dans data/processed/


## 6. Synthèse

- Doublons supprimés (nettoyage minimal, dataset globalement propre)
- Découpage **temporel** 70/15/15 respecté
- Variables temporelles dérivées créées (`TransactionHour`, `TransactionDay`)
- Standardisation fit-train-only validée
- Données prêtes pour `notebooks/03_modelisation.ipynb`